In [3]:
# ============================================================
# EXTRACT CHAT ZIP
# ============================================================

import zipfile
import os

zip_path = "/content/chat.zip"

with zipfile.ZipFile(zip_path, "r") as zip_file:
    zip_file.extractall("/content/chat_data")

print("Files extracted:")

for file_name in os.listdir("/content/chat_data"):
    print(file_name)

Files extracted:
DADS Minor PROJECT dataset.txt


In [7]:
# ============================================================
# GROUPDNA - FEATURE 1: CHAT PARSER
# ============================================================

from datetime import datetime
import os

messages = []

system_messages = 0
media_messages = 0
deleted_messages = 0


# ------------------------------------------------------------
# FIND THE TXT FILE INSIDE chat_data
# ------------------------------------------------------------

chat_file = None

for root, folders, files in os.walk("/content/chat_data"):

    for file_name in files:

        if file_name.lower().endswith(".txt"):

            chat_file = os.path.join(root, file_name)
            break

    if chat_file is not None:
        break


if chat_file is None:

    print("❌ No .txt file found inside chat_data")

else:

    print("Chat file found:", chat_file)

    # --------------------------------------------------------
    # READ THE CHAT
    # --------------------------------------------------------

    with open(chat_file, "r", encoding="utf-8") as file:

        lines = file.readlines()


    # --------------------------------------------------------
    # PARSE MESSAGES
    # --------------------------------------------------------

    for line in lines:

        line = line.strip()

        if line == "":
            continue


        # Check whether line starts with a date
        if (
            len(line) < 8
            or line[2] != "/"
            or line[5] != "/"
        ):

            # Multiline message
            if len(messages) > 0:

                messages[-1]["text"] += " " + line

            continue


        # Separate timestamp and message
        parts = line.split(" - ", 1)

        if len(parts) != 2:
            continue

        timestamp = parts[0]
        remaining = parts[1]


        # Separate sender and text
        parts = remaining.split(": ", 1)


        # System message
        if len(parts) != 2:

            system_messages += 1
            continue


        sender = parts[0]
        text = parts[1]


        # Media message
        if text == "<Media omitted>":

            media_messages += 1
            continue


        # Deleted message
        if text == "This message was deleted":

            deleted_messages += 1
            continue


        # Store message
        messages.append({
            "timestamp": timestamp,
            "sender": sender,
            "text": text
        })


    # --------------------------------------------------------
    # FIND PARTICIPANTS
    # --------------------------------------------------------

    participants = set()

    for message in messages:

        participants.add(message["sender"])


    # --------------------------------------------------------
    # OUTPUT
    # --------------------------------------------------------

    print("\n")
    print("=" * 60)
    print("              GROUPDNA CHAT PARSER")
    print("=" * 60)

    print(
        f"Successfully parsed : {len(messages)} messages"
    )

    print(
        f"Participants        : {len(participants)}"
    )

    print(
        f"System messages     : {system_messages}"
    )

    print(
        f"Media messages      : {media_messages}"
    )

    print(
        f"Deleted messages    : {deleted_messages}"
    )


    print("\nParticipants:")

    for person in participants:

        print(" -", person)


    print("\nFirst 5 messages:")

    for message in messages[:5]:

        print(message)


    print("\nLast 5 messages:")

    for message in messages[-5:]:

        print(message)


    print("=" * 60)

Chat file found: /content/chat_data/DADS Minor PROJECT dataset.txt


              GROUPDNA CHAT PARSER
Successfully parsed : 3127 messages
Participants        : 6
System messages     : 4
Media messages      : 32
Deleted messages    : 15

Participants:
 - Rahul
 - Priya
 - Neha
 - Karan
 - Vikas
 - Aman

First 5 messages:
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'scene fix'}
{'timestamp': '01/04/24, 01:17', 'sender': 'Rahul', 'text': 'haan'}
{'timestamp': '01/04/24, 01:18', 'sender': 'Rahul', 'text': 'kya scene'}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'text': 'abhi free hai?'}
{'timestamp': '01/04/24, 02:13', 'sender': 'Rahul', 'text': 'abey'}

Last 5 messages:
{'timestamp': '30/05/24, 19:14', 'sender': 'Priya', 'text': 'Take care everyone'}
{'timestamp': '30/05/24, 19:28', 'sender': 'Priya', 'text': 'Karan that sounds tough, take care'}
{'timestamp': '30/05/24, 21:17', 'sender': 'Aman', 'text': 'the existential dread is back'}
{'timestamp': '30/05/24, 2

In [12]:
# ============================================================
# GROUPDNA - FEATURE 2: GROUP OVERVIEW
# ============================================================

# ------------------------------------------------------------
# COUNT MESSAGES PER PERSON
# ------------------------------------------------------------

message_counts = {}

for message in messages:

    person = message["sender"]

    if person not in message_counts:
        message_counts[person] = 0

    message_counts[person] += 1


# Sort from highest messages to lowest
ranking = sorted(
    message_counts.items(),
    key=lambda item: item[1],
    reverse=True
)


# ------------------------------------------------------------
# CONVERT TIMESTAMPS TO DATETIME
# ------------------------------------------------------------

dates = []

for message in messages:
    time = datetime.strptime(
        message["timestamp"],
        "%d/%m/%y, %H:%M"
    )
    dates.append(time)


# Find first and last message
first_date = min(dates)
last_date = max(dates)


# Calculate total number of days
total_days = (
    last_date.date() - first_date.date()
).days + 1


# ------------------------------------------------------------
# GROUP OVERVIEW
# ------------------------------------------------------------

print("\n")
print("=" * 65)
print("                    GROUP OVERVIEW")
print("=" * 65)

print(
    f"Total messages : {len(messages)}"
)

print(
    f"Participants   : {len(participants)}"
)

print(
    f"Period         : "
    f"{first_date.strftime('%d %B %Y')} "
    f"to "
    f"{last_date.strftime('%d %B %Y')}"
)

print(
    f"Total days     : {total_days}"
)


# ------------------------------------------------------------
# MESSAGE RANKING
# ------------------------------------------------------------

print("\n")
print("MESSAGES PER PERSON")
print("-" * 65)

for person, count in ranking:

    percentage = (
        count / len(messages)
    ) * 100

    print(
        f"{person:<25}"
        f"{count:>6} messages   "
        f"({percentage:>5.1f}%)"
    )


print("=" * 65)



                    GROUP OVERVIEW
Total messages : 3127
Participants   : 6
Period         : 01 April 2024 to 30 May 2024
Total days     : 60


MESSAGES PER PERSON
-----------------------------------------------------------------
Rahul                       940 messages   ( 30.1%)
Priya                       712 messages   ( 22.8%)
Neha                        624 messages   ( 20.0%)
Aman                        484 messages   ( 15.5%)
Karan                       345 messages   ( 11.0%)
Vikas                        22 messages   (  0.7%)


In [13]:
# ============================================================
# GROUPDNA - FEATURE 3: MOST ACTIVE DAY & HOUR
# ============================================================

day_counts = {}
hour_counts = {}

# Count messages by day and by hour
for message in messages:

    time = datetime.strptime(
        message["timestamp"],
        "%d/%m/%y, %H:%M"
    )

    date = time.date()
    hour = time.hour

    # Count messages for each day
    if date not in day_counts:
        day_counts[date] = 0

    day_counts[date] += 1

    # Count messages for each hour
    if hour not in hour_counts:
        hour_counts[hour] = 0

    hour_counts[hour] += 1


# ------------------------------------------------------------
# FIND BUSIEST DAY
# ------------------------------------------------------------

busiest_day = max(
    day_counts,
    key=day_counts.get
)

busiest_day_count = day_counts[busiest_day]


# ------------------------------------------------------------
# FIND BUSIEST HOUR
# ------------------------------------------------------------

busiest_hour = max(
    hour_counts,
    key=hour_counts.get
)

busiest_hour_count = hour_counts[busiest_hour]


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print("\n")
print("=" * 60)
print("              MOST ACTIVE DAY & HOUR")
print("=" * 60)

print(
    f"Busiest day  : "
    f"{busiest_day.strftime('%d %B %Y')} "
    f"({busiest_day_count} messages)"
)

print(
    f"Busiest hour : "
    f"{busiest_hour:02d}:00 - "
    f"{(busiest_hour + 1) % 24:02d}:00 "
    f"({busiest_hour_count} messages)"
)

print("=" * 60)



              MOST ACTIVE DAY & HOUR
Busiest day  : 04 May 2024 (74 messages)
Busiest hour : 18:00 - 19:00 (244 messages)


In [14]:
# ============================================================
# GROUPDNA - FEATURE 4: ACTIVITY HEATMAP
# ============================================================

import numpy as np

# List of participants
people = sorted(participants)

# Create NumPy matrix
# Rows = participants
# Columns = hours 00 to 23
activity_matrix = np.zeros(
    (len(people), 24),
    dtype=int
)


# ------------------------------------------------------------
# FILL THE MATRIX
# ------------------------------------------------------------

for message in messages:

    time = datetime.strptime(
        message["timestamp"],
        "%d/%m/%y, %H:%M"
    )

    person = message["sender"]
    hour = time.hour

    # Find participant's row
    person_index = people.index(person)

    # Add one message
    activity_matrix[person_index][hour] += 1


# ------------------------------------------------------------
# DISPLAY HEATMAP
# ------------------------------------------------------------

print("\n")
print("=" * 70)
print("                    ACTIVITY HEATMAP")
print("=" * 70)

print("\nMessages by hour of the day\n")

# Hour headings
print(" " * 20, end="")

for hour in range(24):
    print(f"{hour:02d} ", end="")

print()


# ------------------------------------------------------------
# TEXT HEATMAP
# ------------------------------------------------------------

for i in range(len(people)):

    person = people[i]

    maximum = np.max(activity_matrix[i])

    print(f"{person:<20}", end="")

    for hour in range(24):

        value = activity_matrix[i][hour]

        # Avoid division by zero
        if maximum == 0:
            percentage = 0
        else:
            percentage = value / maximum

        # Different activity levels
        if value == 0:
            symbol = "."
        elif percentage <= 0.25:
            symbol = "░"
        elif percentage <= 0.50:
            symbol = "▒"
        elif percentage <= 0.75:
            symbol = "▓"
        else:
            symbol = "█"

        print(f"{symbol}  ", end="")

    print()


# ------------------------------------------------------------
# MATRIX INFORMATION
# ------------------------------------------------------------

print("\n")
print("NumPy matrix:")
print(activity_matrix)

print("\nMatrix shape:", activity_matrix.shape)

print("=" * 70)



                    ACTIVITY HEATMAP

Messages by hour of the day

                    00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Aman                ▓  █  █  ▓  █  .  .  .  .  .  .  .  .  .  ░  ░  ░  ░  ░  ░  ░  ░  .  ▓  
Karan               .  .  .  .  .  .  .  ░  ▒  ▒  ▓  ▒  █  ▓  █  ▓  ▓  ▓  ▓  █  ▓  ▒  ░  ░  
Neha                .  .  .  .  .  ▒  ░  ░  ▓  █  █  ▒  ▓  ▓  ▒  ░  ▓  █  █  █  ▓  ▒  ▒  ▒  
Priya               .  .  .  .  .  .  ░  ▒  ▓  █  █  █  █  ▓  ▓  ▒  ▒  ▓  ▓  █  ▓  ▒  ▒  ░  
Rahul               ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ▓  ▒  ▒  ▓  ▓  ▒  █  ▓  ▒  █  ▓  ▓  
Vikas               .  .  .  .  .  .  .  ▒  ▓  ▒  ▒  .  ▒  ▓  .  ▒  ▒  █  ▓  ▓  ▒  ▒  ▒  ▓  


NumPy matrix:
[[ 53  67  66  60  87   0   0   0   0   0   0   0   0   0  14  11  18   5
   16   8  12  11   0  56]
 [  0   0   0   0   0   0   0   4  11  16  19  16  36  22  32  26  27  27
   24  32  23  14   9   7]
 [  0   0   0   0   0  19   3  13  35  51  52  21  39  36  26 

In [15]:
# ============================================================
# GROUPDNA - FEATURE 5: TOP WORDS
# ============================================================

# Common words that we don't want to count
stop_words = {
    "i", "is", "the", "a", "an", "and", "or",
    "to", "of", "in", "on", "for", "it", "this",
    "that", "you", "me", "my", "we", "are",
    "was", "be", "with", "so", "but", "not",
    "he", "she", "they", "them", "his", "her",
    "was", "were", "have", "has", "had"
}


# Dictionary to store word frequency
word_counts = {}


# ------------------------------------------------------------
# READ EVERY MESSAGE
# ------------------------------------------------------------

for message in messages:

    # Convert message to lowercase
    text = message["text"].lower()

    # Remove punctuation
    punctuation = ".,!?;:'\"()[]{}<>-/\\"

    for symbol in punctuation:
        text = text.replace(symbol, " ")

    # Split into individual words
    words = text.split()


    # --------------------------------------------------------
    # COUNT WORDS
    # --------------------------------------------------------

    for word in words:

        # Ignore empty words
        if word == "":
            continue

        # Ignore stop words
        if word in stop_words:
            continue

        # Add word to dictionary
        if word not in word_counts:
            word_counts[word] = 0

        word_counts[word] += 1


# ------------------------------------------------------------
# SORT WORDS
# ------------------------------------------------------------

top_words = sorted(
    word_counts.items(),
    key=lambda item: item[1],
    reverse=True
)


# ------------------------------------------------------------
# DISPLAY TOP 10
# ------------------------------------------------------------

print("\n")
print("=" * 65)
print("              THIS GROUP'S FAVOURITE WORDS")
print("=" * 65)

if len(top_words) > 0:

    highest_count = top_words[0][1]

    for word, count in top_words[:10]:

        # Create a proportional bar
        bar_length = int(
            (count / highest_count) * 20
        )

        if bar_length < 1:
            bar_length = 1

        bar = "█" * bar_length

        print(
            f"{word:<15} "
            f"{bar:<20} "
            f"{count}"
        )

else:

    print("No words found.")


print("=" * 65)



              THIS GROUP'S FAVOURITE WORDS
how             ████████████████████ 321
guys            ███████████████████  318
today           ██████████████████   292
about           █████████████████    274
hai             ████████████████     268
am              ████████████████     260
at              ████████████████     257
s               ██████████████       226
just            ████████████         208
everyone        ████████████         203


In [16]:
# ============================================================
# GROUPDNA - FEATURE 6: RESPONSE SPEED & SILENT STREAKS
# ============================================================

# ------------------------------------------------------------
# PART A: RESPONSE SPEED
# ------------------------------------------------------------

time_messages = []

for message in messages:

    time = datetime.strptime(
        message["timestamp"],
        "%d/%m/%y, %H:%M"
    )

    time_messages.append({
        "time": time,
        "sender": message["sender"]
    })


# Store response gaps for every person
response_gaps = {}

for person in participants:
    response_gaps[person] = []


# Compare consecutive messages
for i in range(1, len(time_messages)):

    previous_message = time_messages[i - 1]
    current_message = time_messages[i]

    previous_sender = previous_message["sender"]
    current_sender = current_message["sender"]


    # Only count when a different person replies
    if previous_sender != current_sender:

        gap = (
            current_message["time"]
            - previous_message["time"]
        )

        gap_minutes = gap.total_seconds() / 60

        response_gaps[current_sender].append(
            gap_minutes
        )


# Calculate average response time
average_response = {}

for person in participants:

    gaps = response_gaps[person]

    if len(gaps) > 0:

        average_response[person] = (
            sum(gaps) / len(gaps)
        )

    else:

        average_response[person] = None


# ------------------------------------------------------------
# PART B: SILENT STREAKS
# ------------------------------------------------------------

# Get all dates in the chat
all_dates = []

for message in messages:

    time = datetime.strptime(
        message["timestamp"],
        "%d/%m/%y, %H:%M"
    )

    all_dates.append(time.date())


first_day = min(all_dates)
last_day = max(all_dates)


# Total number of days
total_days = (
    last_day - first_day
).days + 1


# Create every date in the chat period
chat_dates = []

for i in range(total_days):

    current_day = first_day.fromordinal(
        first_day.toordinal() + i
    )

    chat_dates.append(current_day)


# ------------------------------------------------------------
# FIND ACTIVE DAYS FOR EACH PERSON
# ------------------------------------------------------------

active_days = {}

for person in participants:

    active_days[person] = set()


for message in messages:

    time = datetime.strptime(
        message["timestamp"],
        "%d/%m/%y, %H:%M"
    )

    active_days[
        message["sender"]
    ].add(time.date())


# ------------------------------------------------------------
# FIND LONGEST SILENT STREAK
# ------------------------------------------------------------

silent_streaks = {}

for person in participants:

    current_streak = 0
    longest_streak = 0

    for day in chat_dates:

        if day not in active_days[person]:

            current_streak += 1

            if current_streak > longest_streak:
                longest_streak = current_streak

        else:

            current_streak = 0


    silent_streaks[person] = longest_streak


# ------------------------------------------------------------
# RESPONSE RANKING
# ------------------------------------------------------------

people_with_response = []

for person in participants:

    if average_response[person] is not None:

        people_with_response.append(person)


# ------------------------------------------------------------
# OUTPUT
# ------------------------------------------------------------

print("\n")
print("=" * 65)
print("                 RESPONSE PATTERNS")
print("=" * 65)


if len(people_with_response) > 0:

    fastest = min(
        people_with_response,
        key=lambda person:
        average_response[person]
    )

    slowest = max(
        people_with_response,
        key=lambda person:
        average_response[person]
    )


    print(
        f"Fastest replier : {fastest}"
    )

    print(
        f"Average response: "
        f"{average_response[fastest]:.1f} minutes"
    )


    print()


    print(
        f"Slowest replier : {slowest}"
    )

    print(
        f"Average response: "
        f"{average_response[slowest]:.1f} minutes"
    )


# ------------------------------------------------------------
# SILENT STREAK RANKING
# ------------------------------------------------------------

print("\nLONGEST SILENT STREAKS")
print("-" * 65)


silent_ranking = sorted(
    silent_streaks.items(),
    key=lambda item: item[1],
    reverse=True
)


for person, streak in silent_ranking:

    print(
        f"{person:<25} : "
        f"{streak} days"
    )


print("=" * 65)



                 RESPONSE PATTERNS
Fastest replier : Vikas
Average response: 34.9 minutes

Slowest replier : Aman
Average response: 54.9 minutes

LONGEST SILENT STREAKS
-----------------------------------------------------------------
Vikas                     : 11 days
Rahul                     : 0 days
Priya                     : 0 days
Neha                      : 0 days
Karan                     : 0 days
Aman                      : 0 days


In [17]:
# ============================================================
# GROUPDNA - FEATURE 7: PERSONALITY ARCHETYPES
# ============================================================


# ------------------------------------------------------------
# STORE MESSAGES PERSON-WISE
# ------------------------------------------------------------

person_messages = {}

for person in participants:
    person_messages[person] = []


for message in messages:

    person_messages[message["sender"]].append(message)


# ------------------------------------------------------------
# 1. THE SPAMMER
# Average consecutive messages
# ------------------------------------------------------------

def spammer_score(person):

    bursts = []
    current_burst = 0

    for message in messages:

        if message["sender"] == person:

            current_burst += 1

        else:

            if current_burst > 0:
                bursts.append(current_burst)

            current_burst = 0


    # Save the final burst
    if current_burst > 0:
        bursts.append(current_burst)


    if len(bursts) == 0:
        return 0


    average_burst = sum(bursts) / len(bursts)


    # PDF threshold = average burst greater than 3
    if average_burst <= 3:
        return 0


    score = ((average_burst - 3) / 3) * 100


    if score > 100:
        score = 100


    return score


# ------------------------------------------------------------
# 2. THE GROUP MOM
# Caring keywords
# ------------------------------------------------------------

def group_mom_score(person):

    caring_words = [
        "okay",
        "safe",
        "eat",
        "sleep",
        "take care",
        "are you",
        "please",
        "reminder",
        "drink water",
        "don't forget"
    ]


    count = 0


    for message in person_messages[person]:

        text = message["text"].lower()


        for word in caring_words:

            if word in text:

                count += 1


    # More caring messages = stronger score
    score = count * 5


    if score > 100:
        score = 100


    return score


# ------------------------------------------------------------
# 3. THE NIGHT OWL
# More than 60% of messages between 23:00 and 04:59
# ------------------------------------------------------------

def night_owl_score(person):

    total = len(person_messages[person])


    if total == 0:
        return 0


    night_messages = 0


    for message in person_messages[person]:

        time = datetime.strptime(
            message["timestamp"],
            "%d/%m/%y, %H:%M"
        )


        if time.hour >= 23 or time.hour <= 4:

            night_messages += 1


    percentage = (
        night_messages / total
    ) * 100


    # PDF threshold = 60%
    if percentage <= 60:
        return 0


    score = ((percentage - 60) / 40) * 100


    if score > 100:
        score = 100


    return score


# ------------------------------------------------------------
# 4. THE STORYTELLER
# Average words per message greater than 30
# ------------------------------------------------------------

def storyteller_score(person):

    total = len(person_messages[person])


    if total == 0:
        return 0


    total_words = 0


    for message in person_messages[person]:

        words = message["text"].split()

        total_words += len(words)


    average_words = total_words / total


    # PDF threshold = 30 words
    if average_words <= 30:
        return 0


    score = ((average_words - 30) / 30) * 100


    if score > 100:
        score = 100


    return score


# ------------------------------------------------------------
# 5. THE DRAMA QUEEN
# More than 30% ALL-CAPS OR 2+ exclamation marks
# ------------------------------------------------------------

def drama_queen_score(person):

    total = len(person_messages[person])


    if total == 0:
        return 0


    drama_messages = 0


    for message in person_messages[person]:

        text = message["text"].strip()


        # Ignore messages shorter than 3 characters
        if len(text) < 3:
            continue


        if text.isupper() or text.count("!") >= 2:

            drama_messages += 1


    percentage = (
        drama_messages / total
    ) * 100


    # PDF threshold = 30%
    if percentage <= 30:
        return 0


    score = ((percentage - 30) / 70) * 100


    if score > 100:
        score = 100


    return score


# ------------------------------------------------------------
# 6. THE GHOST
# Silent on more than 60% of days
# ------------------------------------------------------------

def ghost_score(person):

    active = len(active_days[person])

    silent_days = total_days - active


    percentage = (
        silent_days / total_days
    ) * 100


    # PDF threshold = 60%
    if percentage <= 60:
        return 0


    score = ((percentage - 60) / 40) * 100


    if score > 100:
        score = 100


    return score


# ------------------------------------------------------------
# 7. THE COMEDIAN
# Highest percentage of comedy words
# ------------------------------------------------------------

def comedian_score(person):

    comedy_words = [
        "lol",
        "lmao",
        "haha",
        "rofl",
        "lmfao"
    ]


    total = len(person_messages[person])


    if total == 0:
        return 0


    comedy_count = 0


    for message in person_messages[person]:

        text = message["text"].lower()


        for word in comedy_words:

            if word in text:

                comedy_count += 1
                break


    percentage = (
        comedy_count / total
    ) * 100


    score = percentage * 2


    if score > 100:
        score = 100


    return score


# ------------------------------------------------------------
# 8. THE QUESTION MASTER
# More than 25% messages end with ?
# ------------------------------------------------------------

def question_master_score(person):

    total = len(person_messages[person])


    if total == 0:
        return 0


    questions = 0


    for message in person_messages[person]:

        text = message["text"].strip()


        if text.endswith("?"):

            questions += 1


    percentage = (
        questions / total
    ) * 100


    # PDF threshold = 25%
    if percentage <= 25:
        return 0


    score = ((percentage - 25) / 75) * 100


    if score > 100:
        score = 100


    return score


# ============================================================
# CALCULATE ALL ARCHETYPE SCORES
# ============================================================

archetype_scores = {}


for person in participants:

    archetype_scores[person] = {

        "THE SPAMMER":
            spammer_score(person),

        "THE GROUP MOM":
            group_mom_score(person),

        "THE NIGHT OWL":
            night_owl_score(person),

        "THE STORYTELLER":
            storyteller_score(person),

        "THE DRAMA QUEEN":
            drama_queen_score(person),

        "THE GHOST":
            ghost_score(person),

        "THE COMEDIAN":
            comedian_score(person),

        "THE QUESTION MASTER":
            question_master_score(person)
    }


# ============================================================
# ASSIGN ONE ARCHETYPE TO EACH PERSON
# ============================================================

person_archetypes = {}


for person in participants:

    scores = archetype_scores[person]


    best_archetype = max(
        scores,
        key=scores.get
    )


    person_archetypes[person] = best_archetype


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("                 PERSONALITY ARCHETYPES")
print("=" * 70)


for person in sorted(person_archetypes):

    archetype = person_archetypes[person]

    score = archetype_scores[person][archetype]


    print(
        f"{person:<25} → "
        f"{archetype:<22} "
        f"(score: {score:.1f})"
    )


print("=" * 70)



                 PERSONALITY ARCHETYPES
Aman                      → THE GROUP MOM          (score: 100.0)
Karan                     → THE GROUP MOM          (score: 100.0)
Neha                      → THE GROUP MOM          (score: 100.0)
Priya                     → THE GROUP MOM          (score: 100.0)
Rahul                     → THE SPAMMER            (score: 49.2)
Vikas                     → THE COMEDIAN           (score: 36.4)


In [18]:
# ============================================================
# GROUPDNA - FEATURE 8: FINAL REPORT
# ============================================================

print("\n")
print("=" * 75)
print("                         GROUPDNA REPORT")
print("=" * 75)

print(
    f" Participants : {len(participants)}"
)

print(
    f" Messages     : {len(messages)}"
)

print(
    f" Period       : "
    f"{first_date.strftime('%d %B %Y')} "
    f"to "
    f"{last_date.strftime('%d %B %Y')}"
)

print(
    f" Total days   : {total_days}"
)

print("=" * 75)


# ============================================================
# MOST ACTIVE DAY & HOUR
# ============================================================

print("\nMOST ACTIVE TIME")
print("-" * 75)

print(
    f"Busiest day  : "
    f"{busiest_day.strftime('%d %B %Y')} "
    f"({busiest_day_count} messages)"
)

print(
    f"Busiest hour : "
    f"{busiest_hour:02d}:00 - "
    f"{(busiest_hour + 1) % 24:02d}:00 "
    f"({busiest_hour_count} messages)"
)


# ============================================================
# MESSAGES PER PERSON
# ============================================================

print("\nMESSAGES PER PERSON")
print("-" * 75)

highest_messages = ranking[0][1]

for person, count in ranking:

    percentage = (
        count / len(messages)
    ) * 100

    bar_length = int(
        (count / highest_messages) * 20
    )

    if bar_length < 1:
        bar_length = 1

    bar = "█" * bar_length

    print(
        f"{person:<25} "
        f"{bar:<20} "
        f"{count:>5} "
        f"({percentage:>5.1f}%)"
    )


# ============================================================
# ACTIVITY HEATMAP
# ============================================================

print("\nACTIVITY HEATMAP")
print("(Messages by hour)")

print(" " * 25, end="")

for hour in range(24):

    print(
        f"{hour:02d} ",
        end=""
    )

print()


for i in range(len(people)):

    person = people[i]

    maximum = np.max(
        activity_matrix[i]
    )

    print(
        f"{person:<25}",
        end=""
    )


    for hour in range(24):

        value = activity_matrix[i][hour]


        if maximum == 0:

            percentage = 0

        else:

            percentage = value / maximum


        if value == 0:

            symbol = "."

        elif percentage <= 0.25:

            symbol = "░"

        elif percentage <= 0.50:

            symbol = "▒"

        elif percentage <= 0.75:

            symbol = "▓"

        else:

            symbol = "█"


        print(
            f"{symbol}  ",
            end=""
        )


    print()


# ============================================================
# TOP WORDS
# ============================================================

print("\nTHIS GROUP'S FAVOURITE WORDS")
print("-" * 75)

if len(top_words) > 0:

    highest_word_count = top_words[0][1]


    for word, count in top_words[:10]:

        bar_length = int(
            (count / highest_word_count) * 20
        )


        if bar_length < 1:
            bar_length = 1


        bar = "█" * bar_length


        print(
            f"{word:<15} "
            f"{bar:<20} "
            f"{count}"
        )


# ============================================================
# RESPONSE PATTERNS
# ============================================================

print("\nRESPONSE PATTERNS")
print("-" * 75)

if len(people_with_response) > 0:

    print(
        f"Fastest replier : "
        f"{fastest} "
        f"({average_response[fastest]:.1f} minutes)"
    )

    print(
        f"Slowest replier : "
        f"{slowest} "
        f"({average_response[slowest]:.1f} minutes)"
    )


print("\nLONGEST SILENT STREAKS")

for person, streak in silent_ranking:

    print(
        f"{person:<25} : "
        f"{streak} days"
    )


# ============================================================
# PERSONALITY ARCHETYPES
# ============================================================

print("\nPERSONALITY ARCHETYPES")
print("-" * 75)

for person in sorted(person_archetypes):

    archetype = person_archetypes[person]

    score = archetype_scores[person][archetype]


    print(
        f"{person:<25} → "
        f"{archetype:<23} "
        f"(score: {score:.1f})"
    )


# ============================================================
# FOOTER
# ============================================================

print("\n")
print("=" * 75)
print("             Generated by GroupDNA")
print("                 Python + NumPy")
print("=" * 75)



                         GROUPDNA REPORT
 Participants : 6
 Messages     : 3127
 Period       : 01 April 2024 to 30 May 2024
 Total days   : 60

MOST ACTIVE TIME
---------------------------------------------------------------------------
Busiest day  : 04 May 2024 (74 messages)
Busiest hour : 18:00 - 19:00 (244 messages)

MESSAGES PER PERSON
---------------------------------------------------------------------------
Rahul                     ████████████████████   940 ( 30.1%)
Priya                     ███████████████        712 ( 22.8%)
Neha                      █████████████          624 ( 20.0%)
Aman                      ██████████             484 ( 15.5%)
Karan                     ███████                345 ( 11.0%)
Vikas                     █                       22 (  0.7%)

ACTIVITY HEATMAP
(Messages by hour)
                         00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 
Aman                     ▓  █  █  ▓  █  .  .  .  .  .  .  .  .  .  ░  ░